# Исследовательский анализ данных (EDA): xAPI-Edu-Data

Набор данных об успеваемости и учебной активности студентов в системе электронного обучения.
Блокнот выполняет этапы 1–10 задания: загрузка, описание, поиск и заполнение пропусков, удаление лишних столбцов,
разделение признаков на числовые и категориальные, расчёт статистик с сохранением в CSV, анализ категориальных признаков.

## 1–2. Загрузка данных и проверка корректности чтения столбцов

In [ ]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

df = pd.read_csv('xAPI-Edu-Data.csv')
df.head()

,Unnamed: 0,gender,NationalITy,PlaceofBirth,StageID,GradeID,SectionID,Topic,Semester,Relation,raisedhands,VisITedResources,AnnouncementsView,Discussion,ParentAnsweringSurvey,ParentschoolSatisfaction,StudentAbsenceDays,Class
0,0,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,15,16,2,20,Yes,Good,Under-7,M
1,1,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,20,20,3,25,Yes,Good,Under-7,M
2,2,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,10,7,0,30,No,Bad,Above-7,L
3,3,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,30,25,5,35,No,Bad,Above-7,L
4,4,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,40,50,12,50,No,Bad,Above-7,M


In [ ]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 480 entries, 0 to 479
Data columns (total 18 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Unnamed: 0                480 non-null    int64
 1   gender                    480 non-null    str  
 2   NationalITy               480 non-null    str  
 3   PlaceofBirth              480 non-null    str  
 4   StageID                   480 non-null    str  
 5   GradeID                   480 non-null    str  
 6   SectionID                 480 non-null    str  
 7   Topic                     480 non-null    str  
 8   Semester                  480 non-null    str  
 9   Relation                  480 non-null    str  
 10  raisedhands               480 non-null    int64
 11  VisITedResources          480 non-null    int64
 12  AnnouncementsView         480 non-null    int64
 13  Discussion                480 non-null    int64
 14  ParentAnsweringSurvey     480 non-null    str  
 15  

In [3]:
# столбец 'Unnamed: 0' — подозрение на сохранённый индекс: сверим его со стандартной нумерацией строк
(df['Unnamed: 0'] == np.arange(len(df))).all()

np.True_

**Вывод по чтению данных.** Файл прочитался корректно: разделитель — запятая, все 18 столбцов разобраны,
числовые столбцы имеют тип `int64`, текстовые — строковый тип (`object`/`str`).
Столбец `Unnamed: 0` — это индекс строк, сохранённый при экспорте файла (значения совпадают с `0…n-1`), то есть не признак.
В названиях столбцов есть нестандартный регистр (`NationalITy`, `VisITedResources`) — это особенность исходного датасета, на чтение она не влияет.

## 3. Описание данных

Набор **xAPI-Edu-Data** собран из системы управления обучением Kalboard 360 (Иордания) с помощью API Experience (xAPI).
Каждая строка — один студент. Столбцы делятся на три группы: демографические, академические и поведенческие.

| Столбец | Группа | Описание | Тип |
|---|---|---|---|
| `Unnamed: 0` | служебный | Индекс строки, сохранённый при экспорте | int |
| `gender` | демографический | Пол студента: M — мужской, F — женский | категориальный (бинарный) |
| `NationalITy` | демографический | Национальность (страна гражданства) | категориальный |
| `PlaceofBirth` | демографический | Место рождения (страна) | категориальный |
| `StageID` | академический | Образовательный уровень: lowerlevel, MiddleSchool, HighSchool | категориальный (порядковый) |
| `GradeID` | академический | Класс: G-02 … G-12 | категориальный (порядковый) |
| `SectionID` | академический | Секция (группа) класса: A, B, C | категориальный |
| `Topic` | академический | Учебный предмет (IT, French, Arabic, Math и др.) | категориальный |
| `Semester` | академический | Семестр: F — осенний (first), S — весенний (second) | категориальный (бинарный) |
| `Relation` | академический | Ответственный за студента родитель: Father / Mum | категориальный (бинарный) |
| `raisedhands` | поведенческий | Сколько раз студент поднимал руку на занятии | числовой |
| `VisITedResources` | поведенческий | Сколько раз студент заходил в учебные материалы курса | числовой |
| `AnnouncementsView` | поведенческий | Сколько раз студент просматривал объявления | числовой |
| `Discussion` | поведенческий | Сколько раз студент участвовал в обсуждениях | числовой |
| `ParentAnsweringSurvey` | поведенческий | Отвечал ли родитель на опросы школы: Yes / No | категориальный (бинарный) |
| `ParentschoolSatisfaction` | поведенческий | Удовлетворённость родителя школой: Good / Bad | категориальный (бинарный) |
| `StudentAbsenceDays` | поведенческий | Количество пропусков: Under-7 / Above-7 | категориальный (порядковый, бинарный) |
| `Class` | **целевая переменная** | Итоговый уровень успеваемости: L — низкий, M — средний, H — высокий | категориальный (порядковый) |

In [4]:
# краткая статистика по числовым столбцам и по текстовым столбцам
display(df.describe())
display(df.describe(include='all').T[['count', 'unique', 'top', 'freq']].dropna(subset=['unique']))

,Unnamed: 0,raisedhands,VisITedResources,AnnouncementsView,Discussion
count,480.000000,480.000000,480.000000,480.000000,480.000000
mean,239.500000,46.775000,54.797917,37.918750,43.283333
std,138.708327,30.779223,33.080007,26.611244,27.637735
min,0.000000,0.000000,0.000000,0.000000,1.000000
25%,119.750000,15.750000,20.000000,14.000000,20.000000
50%,239.500000,50.000000,65.000000,33.000000,39.000000
75%,359.250000,75.000000,84.000000,58.000000,70.000000
max,479.000000,100.000000,99.000000,98.000000,99.000000


,count,unique,top,freq
gender,480,2,M,305
NationalITy,480,14,KW,179
PlaceofBirth,480,14,KuwaIT,180
StageID,480,3,MiddleSchool,248
GradeID,480,10,G-02,147
SectionID,480,3,A,283
Topic,480,12,IT,95
Semester,480,2,F,245
Relation,480,2,Father,283
ParentAnsweringSurvey,480,2,Yes,270


## 4. Количество строк и столбцов

In [5]:
n_rows, n_cols = df.shape
print(f'Строк: {n_rows}')
print(f'Столбцов: {n_cols}')

Строк: 480
Столбцов: 18


## 5. Количество пустых ячеек

In [6]:
missing = df.isnull().sum()
print(missing)
print('\nВсего пустых ячеек:', int(missing.sum()))
print('Доля пустых ячеек от всех ячеек:', round(missing.sum() / df.size * 100, 2), '%')

Unnamed: 0                  0
gender                      0
NationalITy                 0
PlaceofBirth                0
StageID                     0
GradeID                     0
SectionID                   0
Topic                       0
Semester                    0
Relation                    0
raisedhands                 0
VisITedResources            0
AnnouncementsView           0
Discussion                  0
ParentAnsweringSurvey       0
ParentschoolSatisfaction    0
StudentAbsenceDays          0
Class                       0
dtype: int64

Всего пустых ячеек: 0
Доля пустых ячеек от всех ячеек: 0.0 %


In [7]:
# дополнительная проверка: «замаскированные» пропуски — пустые строки и типичные заглушки
placeholders = ['', ' ', '?', 'NA', 'N/A', 'nan', 'NaN', 'None', 'null', '-']
print('Заглушек вместо пропусков:', int(df.isin(placeholders).sum().sum()))
print('Полных дубликатов строк:', int(df.duplicated().sum()))

Заглушек вместо пропусков: 0
Полных дубликатов строк: 0


**Вывод.** В данных **нет пустых ячеек** (0 из 8640), нет и скрытых пропусков в виде заглушек, нет дубликатов строк.
Тем не менее задание требует реализовать заполнение пропусков, поэтому ниже оформлена универсальная функция
и показана её работа на копии данных с искусственно внесёнными пропусками.

## 6. Заполнение пустых ячеек

Логика замещения:
- **числовые признаки** — медиана столбца (устойчива к выбросам);
- **категориальные признаки** — мода (самое частое значение).

In [8]:
def fill_missing(data):
    """Заполняет пропуски: медиана для числовых столбцов, мода для остальных."""
    data = data.copy()
    for col in data.columns:
        if data[col].isnull().any():
            if pd.api.types.is_numeric_dtype(data[col]):
                data[col] = data[col].fillna(data[col].median())
            else:
                data[col] = data[col].fillna(data[col].mode()[0])
    return data

# --- Применяем к реальным данным (пропусков нет, поэтому данные не изменятся) ---
df_filled = fill_missing(df)
print('Пустых ячеек после заполнения:', int(df_filled.isnull().sum().sum()))

Пустых ячеек после заполнения: 0


In [9]:
# --- Демонстрация на копии: вносим 5% случайных пропусков и проверяем работу функции ---
rng = np.random.default_rng(42)
demo = df.copy().astype(object)
mask = rng.random(demo.shape) < 0.05
demo = demo.mask(mask)
# возвращаем числовым столбцам числовой тип
for col in ['raisedhands', 'VisITedResources', 'AnnouncementsView', 'Discussion']:
    demo[col] = pd.to_numeric(demo[col])

print('Пропусков в демонстрационной копии:', int(demo.isnull().sum().sum()))
demo_filled = fill_missing(demo)
print('Пропусков после заполнения:       ', int(demo_filled.isnull().sum().sum()))

Пропусков в демонстрационной копии: 430
Пропусков после заполнения:        0


## 7. Удаление столбцов, не являющихся признаками, и столбцов с большим числом пропусков

- `Unnamed: 0` — индекс строк, не несёт информации о студенте → удаляем.
- Столбцы, где пропущено более 40 % значений, → удаляем (в данных таких нет).
- `Class` — целевая переменная. Она не входит в «признаки» в строгом смысле, но нужна для обучения модели, поэтому **оставляем** и дальше отмечаем как target.

In [ ]:
# служебный столбец-индекс
df_clean = df_filled.drop(columns=['Unnamed: 0'])

# столбцы с долей пропусков > 40 % (считаем по исходным данным, до заполнения)
THRESHOLD = 0.4
share_missing = df.isnull().mean()
too_empty = share_missing[share_missing > THRESHOLD].index.tolist()
print('Столбцы с долей пропусков >', THRESHOLD, ':', too_empty)
df_clean = df_clean.drop(columns=[c for c in too_empty if c in df_clean.columns])

print('Размер после очистки:', df_clean.shape)
df_clean.columns.tolist()

Столбцы с долей пропусков > 0.4 : []
Размер после очистки: (480, 17)


['gender',
 'NationalITy',
 'PlaceofBirth',
 'StageID',
 'GradeID',
 'SectionID',
 'Topic',
 'Semester',
 'Relation',
 'raisedhands',
 'VisITedResources',
 'AnnouncementsView',
 'Discussion',
 'ParentAnsweringSurvey',
 'ParentschoolSatisfaction',
 'StudentAbsenceDays',
 'Class']

## 8. Разделение признаков на числовые и категориальные

In [11]:
num_df = df_clean.select_dtypes(include='number')
cat_df = df_clean.select_dtypes(exclude='number')   # включает целевую переменную Class

print('Числовые признаки    :', num_df.columns.tolist())
print('Категориальные       :', cat_df.columns.tolist())
print('Всего:', num_df.shape[1] + cat_df.shape[1], '=', df_clean.shape[1])

Числовые признаки    : ['raisedhands', 'VisITedResources', 'AnnouncementsView', 'Discussion']
Категориальные       : ['gender', 'NationalITy', 'PlaceofBirth', 'StageID', 'GradeID', 'SectionID', 'Topic', 'Semester', 'Relation', 'ParentAnsweringSurvey', 'ParentschoolSatisfaction', 'StudentAbsenceDays', 'Class']
Всего: 17 = 17


In [12]:
num_df.head()

,raisedhands,VisITedResources,AnnouncementsView,Discussion
0,15,16,2,20
1,20,20,3,25
2,10,7,0,30
3,30,25,5,35
4,40,50,12,50


In [13]:
cat_df.head()

,gender,NationalITy,PlaceofBirth,StageID,GradeID,SectionID,Topic,Semester,Relation,ParentAnsweringSurvey,ParentschoolSatisfaction,StudentAbsenceDays,Class
0,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,Yes,Good,Under-7,M
1,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,Yes,Good,Under-7,M
2,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,No,Bad,Above-7,L
3,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,No,Bad,Above-7,L
4,M,KW,KuwaIT,lowerlevel,G-04,A,IT,F,Father,No,Bad,Above-7,M


## 9. Аналитика по признакам и сохранение в CSV

**a. Количественные признаки:** максимум, минимум, среднее, медиана, дисперсия (выборочная, `ddof=1`), квантили 0.1 / 0.9 и 0.25 / 0.75.

In [14]:
num_stats = pd.DataFrame({
    'max':      num_df.max(),
    'min':      num_df.min(),
    'mean':     num_df.mean(),
    'median':   num_df.median(),
    'variance': num_df.var(),
    'q0.10':    num_df.quantile(0.10),
    'q0.90':    num_df.quantile(0.90),
    'q0.25':    num_df.quantile(0.25),
    'q0.75':    num_df.quantile(0.75),
}).round(4)
num_stats.index.name = 'feature'

num_stats.to_csv('numeric_stats.csv')
num_stats

,max,min,mean,median,variance,q0.10,q0.90,q0.25,q0.75
feature,,,,,,,,,
raisedhands,100,0,46.7750,50.0,947.3605,9.0,87.0,15.75,75.0
VisITedResources,99,0,54.7979,65.0,1094.2868,7.0,90.0,20.00,84.0
AnnouncementsView,98,0,37.9188,33.0,708.1583,5.0,79.1,14.00,58.0
Discussion,99,1,43.2833,39.0,763.8444,10.0,84.0,20.00,70.0


**b. Категориальные признаки:** количество уникальных значений и мода (для наглядности добавлены частота и доля моды).

In [15]:
cat_stats = pd.DataFrame({
    'n_unique':  cat_df.nunique(),
    'mode':      cat_df.mode().iloc[0],
    'mode_freq': [cat_df[c].value_counts().iloc[0] for c in cat_df.columns],
})
cat_stats['mode_share_%'] = (cat_stats['mode_freq'] / len(cat_df) * 100).round(2)
cat_stats.index.name = 'feature'

cat_stats.to_csv('categorical_stats.csv')
cat_stats

,n_unique,mode,mode_freq,mode_share_%
feature,,,,
gender,2,M,305,63.54
NationalITy,14,KW,179,37.29
PlaceofBirth,14,KuwaIT,180,37.50
StageID,3,MiddleSchool,248,51.67
GradeID,10,G-02,147,30.63
SectionID,3,A,283,58.96
Topic,12,IT,95,19.79
Semester,2,F,245,51.04
Relation,2,Father,283,58.96


In [16]:
# проверим, что файлы записались и читаются
print(pd.read_csv('numeric_stats.csv').shape, pd.read_csv('categorical_stats.csv').shape)

(4, 10) (13, 5)


In [17]:
# значения категориальных признаков — нужны для ответа на вопрос 10
for col in cat_df.columns:
    print(f'{col}:', cat_df[col].value_counts().to_dict())

gender: {'M': 305, 'F': 175}
NationalITy: {'KW': 179, 'Jordan': 172, 'Palestine': 28, 'Iraq': 22, 'lebanon': 17, 'Tunis': 12, 'SaudiArabia': 11, 'Egypt': 9, 'Syria': 7, 'USA': 6, 'Iran': 6, 'Lybia': 6, 'Morocco': 4, 'venzuela': 1}
PlaceofBirth: {'KuwaIT': 180, 'Jordan': 176, 'Iraq': 22, 'lebanon': 19, 'SaudiArabia': 16, 'USA': 16, 'Palestine': 10, 'Egypt': 9, 'Tunis': 9, 'Iran': 6, 'Syria': 6, 'Lybia': 6, 'Morocco': 4, 'venzuela': 1}
StageID: {'MiddleSchool': 248, 'lowerlevel': 199, 'HighSchool': 33}
GradeID: {'G-02': 147, 'G-08': 116, 'G-07': 101, 'G-04': 48, 'G-06': 32, 'G-11': 13, 'G-12': 11, 'G-09': 5, 'G-10': 4, 'G-05': 3}
SectionID: {'A': 283, 'B': 167, 'C': 30}
Topic: {'IT': 95, 'French': 65, 'Arabic': 59, 'Science': 51, 'English': 45, 'Biology': 30, 'Spanish': 25, 'Chemistry': 24, 'Geology': 24, 'Quran': 22, 'Math': 21, 'History': 19}
Semester: {'F': 245, 'S': 235}
Relation: {'Father': 283, 'Mum': 197}
ParentAnsweringSurvey: {'Yes': 270, 'No': 210}
ParentschoolSatisfaction: {'G

## 10. Есть ли категориальные признаки, требующие преобразования?

**Да.** Все 12 категориальных признаков (и целевая переменная `Class`) хранятся в виде текста, а большинство алгоритмов машинного обучения (в том числе `scikit-learn`) работают только с числами, поэтому их нужно закодировать. Способ зависит от типа признака:

| Тип | Признаки | Рекомендуемое преобразование |
|---|---|---|
| Бинарные | `gender`, `Semester`, `Relation`, `ParentAnsweringSurvey`, `ParentschoolSatisfaction` | Кодирование 0/1 (или `get_dummies(drop_first=True)`) |
| Порядковые | `StageID`, `GradeID`, `StudentAbsenceDays` | Порядковое кодирование с сохранением порядка (например, `lowerlevel < MiddleSchool < HighSchool`, `Under-7 < Above-7`) |
| Номинальные | `NationalITy`, `PlaceofBirth`, `Topic`, `SectionID` | One-hot encoding (`pd.get_dummies`) |
| Целевая | `Class` | Порядковое кодирование `L=0, M=1, H=2` (или `LabelEncoder`) |

**Что нужно учесть перед кодированием:**
1. **Опечатки и несогласованность написания.** В `NationalITy` и `PlaceofBirth` встречаются `lebanon` (строчная буква), `venzuela`, `Lybia`, `Tunis`, `SaudiArabia`, а Кувейт записан как `KW` в одном столбце и `KuwaIT` в другом. Без унификации один и тот же объект станет разными категориями.
2. **Высокая кардинальность и редкие категории.** В `NationalITy` и `PlaceofBirth` по 14 значений, многие из которых встречаются 1–6 раз; one-hot даст много разреженных столбцов. Редкие значения можно объединить в категорию `Other`.
3. **Дублирование информации.** `NationalITy` и `PlaceofBirth` сильно пересекаются, а `GradeID` однозначно определяет `StageID` — возможна мультиколлинеарность, один из таких столбцов можно исключить.
4. **Dummy-ловушка.** При one-hot нужно удалять один столбец (`drop_first=True`), как в примере с Титаником.
5. **Числовые признаки** (`raisedhands`, `VisITedResources` и др.) — шкалы 0–100, их можно дополнительно нормализовать (`StandardScaler`).

In [ ]:
# Демонстрация преобразования
enc = df_clean.copy()

# унификация написания
fix = {'KW': 'Kuwait', 'KuwaIT': 'Kuwait', 'lebanon': 'Lebanon', 'venzuela': 'Venezuela',
       'Lybia': 'Libya', 'Tunis': 'Tunisia', 'SaudiArabia': 'Saudi Arabia'}
enc[['NationalITy', 'PlaceofBirth']] = enc[['NationalITy', 'PlaceofBirth']].replace(fix)

# бинарные признаки -> 0/1
binary_maps = {
    'gender': {'F': 0, 'M': 1},
    'Semester': {'F': 0, 'S': 1},
    'Relation': {'Mum': 0, 'Father': 1},
    'ParentAnsweringSurvey': {'No': 0, 'Yes': 1},
    'ParentschoolSatisfaction': {'Bad': 0, 'Good': 1},
}
for col, m in binary_maps.items():
    enc[col] = enc[col].map(m)

# порядковые признаки и целевая переменная
enc['StageID'] = enc['StageID'].map({'lowerlevel': 0, 'MiddleSchool': 1, 'HighSchool': 2})
enc['GradeID'] = enc['GradeID'].str.replace('G-', '', regex=False).astype(int)
enc['StudentAbsenceDays'] = enc['StudentAbsenceDays'].map({'Under-7': 0, 'Above-7': 1})
enc['Class'] = enc['Class'].map({'L': 0, 'M': 1, 'H': 2})

# номинальные признаки -> one-hot
enc = pd.get_dummies(enc, columns=['NationalITy', 'PlaceofBirth', 'Topic', 'SectionID'],
                     drop_first=True, dtype=int)

print('Было столбцов:', df_clean.shape[1], '→ стало:', enc.shape[1])
print('Остались нечисловые столбцы:', enc.select_dtypes(exclude='number').columns.tolist())
enc.head(3)

Было столбцов: 17 → стало: 52
Остались нечисловые столбцы: []


,gender,StageID,GradeID,Semester,Relation,raisedhands,VisITedResources,AnnouncementsView,Discussion,ParentAnsweringSurvey,ParentschoolSatisfaction,StudentAbsenceDays,Class,NationalITy_Iran,NationalITy_Iraq,NationalITy_Jordan,NationalITy_Kuwait,NationalITy_Lebanon,NationalITy_Libya,NationalITy_Morocco,NationalITy_Palestine,NationalITy_Saudi Arabia,NationalITy_Syria,NationalITy_Tunisia,NationalITy_USA,NationalITy_Venezuela,PlaceofBirth_Iran,PlaceofBirth_Iraq,PlaceofBirth_Jordan,PlaceofBirth_Kuwait,PlaceofBirth_Lebanon,PlaceofBirth_Libya,PlaceofBirth_Morocco,PlaceofBirth_Palestine,PlaceofBirth_Saudi Arabia,PlaceofBirth_Syria,PlaceofBirth_Tunisia,PlaceofBirth_USA,PlaceofBirth_Venezuela,Topic_Biology,Topic_Chemistry,Topic_English,Topic_French,Topic_Geology,Topic_History,Topic_IT,Topic_Math,Topic_Quran,Topic_Science,Topic_Spanish,SectionID_B,SectionID_C
0,1,0,4,0,1,15,16,2,20,1,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
1,1,0,4,0,1,20,20,3,25,1,1,0,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
2,1,0,4,0,1,10,7,0,30,0,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0,0
